Importacion para evitar problemas de paths relativos.

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

Importación e inicializacion de dependencias y servicios.

In [ ]:
import pandas as pd
from src.Orchestration.google_play_service import GooglePlayService
from src.Orchestration.path_helper import PathHelper

# Data Access
from src.DataAccess.Ingestion.google_play_wrapper import GooglePlayScraper
from src.DataAccess.Refinary.google_review_cleaner import GoogleReviewCleaner
from src.DataAccess.Storage.csv_review_repository import CSVReviewRepository

# Modeling
from src.Models.roberta_sentiment_model import RobertaSentimentModel
from src.Models.flan_t5_large_model import FlanT5LargeModel


scrapper = GooglePlayScraper(lang="es", country="ar")
cleaner = GoogleReviewCleaner()
csv_repository = CSVReviewRepository()

sentiment_model = RobertaSentimentModel()
flan_t5_model = FlanT5LargeModel()

path_helper = PathHelper()

google_service = GooglePlayService(
    scrapper=scrapper,
    cleaner=cleaner,
    repository=csv_repository,
    sentiment_model=sentiment_model,
    flan_t5_model=flan_t5_model,
    path_helper=path_helper,
)

In [ ]:
import datetime

timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S") #estrictamente no es un timestamp, pero es un formato de fecha y hora que se puede usar para identificar la versión de los datos descargados
app_id = "com.mercadolibre"

Primeramente el pipeline rootea los parámetros a utilizar a lo largo de su ejecucion.
Estos son:
* **app_id**: es el ID de Googlela aplicación a analizar en GooglePlay)
* **timestamp**: es el dia y hora en la cual se ejecuto el pipeline, sirve como una suerte de correlation_id para relacionar todos los archivos generados en dicha ejecución, de manera que el nombre de los archivos tendrán el formato  appid_timestamp_execution_step (whatsapp_20263181135_CLEAN).

In [ ]:

google_service.path_helper.set_filepath_components(
    app_id=app_id,
    timestamp=timestamp
)

Obtención de datos.
El servicicio se encarga de llamar a la capa que realiza la ingesta mediante la librería google-play-scraper y delegar la persistencia de los resultados en un archivo ..._*RAW*.csv 

In [ ]:
reviews_df = google_service.collect_and_store_reviews(app_id=app_id, limit=100)

print(reviews_df.head())

Ahora se procede a realizar el limpiado de los datos. El orquestador delega esta tarea al **GoogleReviewCleaner** y posteriormente realiza la persistencia de los datos ya curados. El método *clean_reviews* no genera un nuevo dataframe para evitar realizar una nueva copia, en su lugar modificia el dataframe provisto y lo retorna con dichas modificaciones, previo a persistir dichos cambios en un nuevo archivo con el prefijo ..._*CLEANED*.csv.

In [ ]:
cleaned_reviews_df = google_service.clean_reviews(reviews_df)
print(cleaned_reviews_df.head())

El siguiente paso en el pipeline es agregar el análisis de sentimientos, esto se delega a **RobertaSentimentModel**.
Nuevamente se reutiliza el dataframe para ahorrar alocación de memoria y posteriormente se guarda el archivo con su prefijo correspondiente ..._*ANALYZED*.csv

In [ ]:
sentiment_df = google_service.analyze_sentiment(cleaned_reviews_df)
print(sentiment_df.head())

Finalmente el pipeline manda a ejecutar el modelo alojado en **FlanT5LargeModel**, el cual debiera realizar el resumen de las reseñas (con su sentimiento ya asignado) y seleccionar los puntos mas positivos (negativos).
Este método sí retorna un nuevo dataframe, el cual es almacenado en un archivo ..._*SUMMARY*.csv

In [ ]:
summary_df = google_service.build_summary(sentiment_df)
print(summary_df.head())

Toda esta perorata queda resumida dentro del método *run_pipeline*. El cual ejecuta todos los pasos previos.